In [1]:
from pyspark.sql import SparkSession, functions as F

In [2]:
spark = SparkSession.builder \
.appName("Define a Custom Schema") \
.master("local[2]") \
.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/24 08:19:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
! wget -O /opt/examples/datasets/OnlineRetail2.csv \
https://raw.githubusercontent.com/erkansirin78/datasets/refs/heads/master/OnlineRetail2.csv

--2026-10-24 08:19:56--  https://raw.githubusercontent.com/erkansirin78/datasets/refs/heads/master/OnlineRetail2.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48039726 (46M) [text/plain]
Saving to: ‘/opt/examples/datasets/OnlineRetail2.csv’

/opt/examples/datas 100%[===================>]  45.81M  17.7MB/s    in 2.6s    

2026-10-24 08:20:00 (17.7 MB/s) - ‘/opt/examples/datasets/OnlineRetail2.csv’ saved [48039726/48039726]



In [4]:
! ls -l /opt/examples/datasets/

total 46924
-rw-r--r-- 1 ssh_train users     4365 Oct 24 08:14 Mall_Customers.csv
-rw-r--r-- 1 root      root  48039726 Oct 24 08:20 OnlineRetail2.csv


In [5]:
df = spark.read.format("csv") \
.option("sep", ",") \
.option("header", True) \
.option("inferSchema", True) \
.load("file:///opt/examples/datasets/OnlineRetail2.csv")

In [6]:
print(df.count())

541909


In [8]:
df.limit(3).toPandas()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom


In [10]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [22]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, DoubleType, FloatType, TimestampType
schema1 = StructType(
[
StructField('InvoiceNo', StringType(), True), 
StructField('StockCode', StringType(), True), 
StructField('Description', StringType(), True), 
StructField('Quantity', IntegerType(), True), 
StructField('InvoiceDate', TimestampType(), True), 
StructField('UnitPrice', FloatType(), True), 
StructField('CustomerID', DoubleType(), True), 
StructField('Country', StringType(), True)
]

)

In [23]:
df2 = spark.read.format("csv") \
.option("sep", ",") \
.option("header", True) \
.schema(schema1) \
.load("file:///opt/examples/datasets/OnlineRetail2.csv")

In [24]:
df2.limit(3).toPandas()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom


In [25]:
df2.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: float (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [30]:
sql_way_schema = """InvoiceNo string,
StockCode string,
Description string, 
Quantity integer, 
InvoiceDate timestamp,
UnitPrice float,
CustomerID double,
Country string"""

In [31]:
df3 = spark.read.format("csv") \
.option("sep", ",") \
.option("header", True) \
.schema(sql_way_schema) \
.load("file:///opt/examples/datasets/OnlineRetail2.csv")

In [32]:
df3.show(3)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 3 rows


In [33]:
df3.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: float (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [34]:
spark.stop()